# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library following the Croissant schema specification.

### Dataset Source
The dataset is described by a Croissant schema available at:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and record sets from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load Dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Citation: {getattr(metadata, 'citeAs', None)}\n")
print(f"Spatial Coverage: {getattr(metadata, 'spatialCoverage', None)}\n")
print(f"Temporal Coverage: {getattr(metadata, 'temporalCoverage', None)}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview
List all available record sets and their field definitions (`@id`). Every entity in Croissant, including record sets and fields, is referenced by its `@id`.

Below we enumerate the record set `@id`s and for each, its field `@id`s.

In [ ]:
# List available record sets and their fields by @id

record_set_infos = []
for rs in dataset.record_sets:
    info = {}
    info['@id'] = rs['@id']
    info['name'] = rs.get('name', '(no name)')
    info['description'] = rs.get('description', '(no description)')
    fields = rs.get('field', [])
    # Ensure fields is a list
    if isinstance(fields, dict):
        fields = [fields]
    info['fields'] = [f['@id'] for f in fields if isinstance(f, dict) and '@id' in f]
    record_set_infos.append(info)

print("Available Record Sets:")
for rs in record_set_infos:
    print(f"- Record Set '@id': {rs['@id']}")
    print(f"  Name: {rs['name']}")
    print(f"  Description: {rs['description']}")
    print(f"  Fields by @id: {rs['fields'] if rs['fields'] else '(No fields detected)'}\n")

# For demonstration, print a sample record (if any record set exists)
if record_set_infos:
    example_record_set_id = record_set_infos[0]['@id']
    try:
        for i, rec in enumerate(dataset.records(record_set=example_record_set_id)):
            print(f"Sample record from '{example_record_set_id}':\n", json.dumps(rec, indent=2))
            break
    except Exception as e:
        print(f"Could not extract records: {e}")

## 3. Data Extraction
Load one or more record sets into Pandas DataFrames for analysis. Use the `@id`s identified previously.

This code extracts all record sets listed above as separate DataFrames in a dictionary keyed by their `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_set_infos]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set '{rs_id}' with {len(df)} records.")
    except Exception as e:
        print(f"Failed to load records for '{rs_id}': {e}")

# Show the columns of the first (or an example) record set
if dataframes:
    example_rs_id = record_set_ids[0]
    print(f"\nFields (DataFrame columns) in record set '@id' '{example_rs_id}':")
    print(dataframes[example_rs_id].columns.tolist())
    dataframes[example_rs_id].head()
else:
    print('No record sets were extracted.')

## 4. Exploratory Data Analysis (EDA)
We now demonstrate typical data processing, such as filtering, normalization, and grouping.

***Note:***
Please customize the fields used below based on the actual field `@id`s in your chosen record set!

In [ ]:
# Example EDA: Filter and normalize a numeric field, group by a categorical field
# 1. Choose a record set and fields (update @ids as needed)

if dataframes:
    # Example: Use the first available record set
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]

    # Example: Try to auto-detect numeric and categorical fields
    possible_numeric = [col for col in df.columns if df[col].dtype in ('float64', 'int64')] or df.select_dtypes('number').columns.tolist()
    possible_categorical = [col for col in df.columns if df[col].dtype == object and col != possible_numeric[0] if possible_numeric else True]

    # Pick example fields by @id
    if possible_numeric:
        numeric_field_id = possible_numeric[0]
        print(f"Using numeric field '@id': {numeric_field_id}")
    else:
        print("No numeric field detected.")
        numeric_field_id = df.columns[0]  # fallback

    if possible_categorical:
        group_field_id = possible_categorical[0]
        print(f"Using categorical field '@id': {group_field_id}")
    else:
        print("No categorical field detected.")
        group_field_id = df.columns[0]  # fallback

    # Set a threshold for filtering (example: mean)
    try:
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized field for '{numeric_field_id}':")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by the categorical/group field
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} grouped by '{group_field_id}':")
            print(grouped_df.head())
        else:
            print(f"Field '{group_field_id}' not found for grouping.")
    except Exception as ex:
        print(f"Could not perform EDA: {ex}")
else:
    print('No DataFrames available. Skipping EDA.')

## 5. Visualization
Visualize field distributions or relationships with Matplotlib and Seaborn.

We plot the distribution of the numeric field and, optionally, its mean grouped by the categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field_id' in locals():
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[numeric_field_id], kde=True)
        plt.title(f"Distribution of '{numeric_field_id}' (filtered)")
        plt.xlabel(numeric_field_id)
        plt.show()

        # Plot grouped means if possible
        if 'grouped_df' in locals() and group_field_id in grouped_df.columns:
            plt.figure(figsize=(8,4))
            sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
            plt.xticks(rotation=45)
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.tight_layout()
            plt.show()
    else:
        print('No filtered EDA data for visualization.')
else:
    print('No DataFrames available. Skipping visualization.')

## 6. Conclusion

In this notebook, we demonstrated how to load and explore the FAIR² dataset using the `mlcroissant` library by referencing all entities via their `@id` per the Croissant schema. We provided steps for inspecting metadata, listing record sets and fields (by `@id`), loading data into DataFrames, performing filtering and normalization, and plotting distributions.

For further analysis, refer to the actual field and record set `@id`s from Croissant metadata for advanced EDA or ML tasks.
